In [0]:
# Databricks notebook source
# ops/quality_gate.py

from pyspark.sql import functions as F

# 1. Parameter widgets
dbutils.widgets.text("catalog", "meridian_t02", "Target Catalog")
catalog = dbutils.widgets.get("catalog")

spark.sql(f"USE CATALOG {catalog}")

# 2. Ensure the quality audit results table exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.data_quality_results (
    check_name STRING,
    target_table STRING,
    check_status STRING,
    metric_value BIGINT,
    details STRING,
    evaluated_at TIMESTAMP
)
COMMENT 'Audit log recording automated Lakeflow Job quality gate outcomes'
""")

# 3. Define Assertion Checks
checks_failed = []

def run_check(check_name, target_table, condition_sql, fail_threshold=0, check_type="max_allowed"):
    """
    Executes a quality assertion SQL query, records results into ops, 
    and tracks failure status.
    """
    df = spark.sql(condition_sql)
    val = df.collect()[0][0]
    
    if check_type == "max_allowed":
        passed = (val <= fail_threshold)
        detail = f"Observed: {val}, Maximum Allowed: {fail_threshold}"
    elif check_type == "min_required":
        passed = (val >= fail_threshold)
        detail = f"Observed: {val}, Minimum Required: {fail_threshold}"
    else:
        passed = False
        detail = "Unknown check type"

    status = "PASSED" if passed else "FAILED"

    # Log to audit table
    spark.sql(f"""
    INSERT INTO {catalog}.ops.data_quality_results 
    VALUES ('{check_name}', '{target_table}', '{status}', {val}, '{detail}', current_timestamp())
    """)

    if not passed:
        checks_failed.append(f"[{check_name}] on {target_table} FAILED: {detail}")

# --- Check 1: Empty Gold Check (fct_sessions must contain rows) ---
run_check(
    check_name="empty_table_check",
    target_table="gold.fct_sessions",
    condition_sql=f"SELECT COUNT(*) FROM {catalog}.gold.fct_sessions",
    fail_threshold=1,
    check_type="min_required"
)

# --- Check 2: Uniqueness Check (Zero duplicate session_ids in fct_sessions) ---
run_check(
    check_name="duplicate_primary_key_check",
    target_table="gold.fct_sessions",
    condition_sql=f"""
        SELECT COUNT(*) - COUNT(DISTINCT session_id) 
        FROM {catalog}.gold.fct_sessions
    """,
    fail_threshold=0,
    check_type="max_allowed"
)

# --- Check 3: Orphan Foreign Key Check (All session user_ids exist in dim_users) ---
run_check(
    check_name="orphan_user_foreign_key_check",
    target_table="gold.fct_sessions",
    condition_sql=f"""
        SELECT COUNT(*) 
        FROM {catalog}.gold.fct_sessions s
        LEFT JOIN {catalog}.gold.dim_users u ON s.user_id = u.user_id
        WHERE u.user_id IS NULL
    """,
    fail_threshold=0,
    check_type="max_allowed"
)

# --- Check 4: Positive Financials Check (fct_bookings total_amount must be positive) ---
run_check(
    check_name="non_positive_booking_amount_check",
    target_table="gold.fct_bookings",
    condition_sql=f"""
        SELECT COUNT(*) 
        FROM {catalog}.gold.fct_bookings 
        WHERE total_amount <= 0
    """,
    fail_threshold=0,
    check_type="max_allowed"
)

# 4. Evaluate Overall Gate Status
if len(checks_failed) > 0:
    error_summary = "\n".join(checks_failed)
    print(f"Data Quality Gate Failed:\n{error_summary}")
    raise Exception(f"QUALITY GATE FAILURE: {len(checks_failed)} assertions failed:\n{error_summary}")
else:
    print("All quality checks PASSED successfully.")